In [ ]:
# ============================================================
# SISTEMA DE PREDICCIÓN E INTELIGENCIA DEPORTIVA
# FIFA World Cup 2026
# ============================================================
# Este notebook realiza el análisis exploratorio (EDA) y 
# preprocesamiento de los datasets disponibles para construir
# el sistema de predicción de partidos del Mundial 2026.
# Cada celda incluye comentarios explicando qué se hace y por qué.
# ============================================================

In [2]:
import pandas as pd
import numpy as np
import os

# ============================================================
# CARGA DE DATASETS
# Cargamos cada archivo para ver qué tenemos exactamente
# ============================================================

# Rutas
PATH_RESULTS    = "../data/raw/results_1872_2024/results.csv"
PATH_SHOOTOUTS  = "../data/raw/results_1872_2024/shootouts.csv"
PATH_SCORERS    = "../data/raw/results_1872_2024/goalscorers.csv"
PATH_NAMES      = "../data/raw/results_1872_2024/former_names.csv"
PATH_RANKING_1  = "../data/raw/fifa_world_ranking/fifa_ranking-2023-07-20.csv"
PATH_RANKING_2  = "../data/raw/fifa_world_ranking/fifa_ranking-2024-04-04.csv"
PATH_RANKING_3  = "../data/raw/fifa_world_ranking/fifa_ranking-2024-06-20.csv"
PATH_PLAYERS    = "../data/raw/players_dataset/fifa_world_cup_2026_player_performance.csv"
PATH_ELO        = "../data/raw/world_cup_historical_stats_48_teams/elo_ratings_wc2026.csv"

# Cargamos todos
results   = pd.read_csv(PATH_RESULTS)
shootouts = pd.read_csv(PATH_SHOOTOUTS)
scorers   = pd.read_csv(PATH_SCORERS)
names     = pd.read_csv(PATH_NAMES)
ranking1  = pd.read_csv(PATH_RANKING_1)
ranking2  = pd.read_csv(PATH_RANKING_2)
ranking3  = pd.read_csv(PATH_RANKING_3)
players   = pd.read_csv(PATH_PLAYERS)
elo       = pd.read_csv(PATH_ELO)

In [10]:
# ============================================================
# EXPLORACIÓN INICIAL DE CADA DATASET
# Vemos dimensiones, columnas y primeras filas de cada uno
# ============================================================

datasets = {
    "results"  : results,
    "shootouts": shootouts,
    "scorers"  : scorers,
    "names"    : names,
    "ranking1" : ranking1,
    "ranking2" : ranking2,
    "ranking3" : ranking3,
    "players"  : players,
    "elo"      : elo
}

for nombre, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {nombre}")
    print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Columnas: {list(df.columns)}")
    print(f"\nPrimeras 2 filas:")
    print(df.head(2))
    print(f"\nTipos de datos:")
    print(df.dtypes)
    print(f"\nValores nulos:")
    print(df.isnull().sum())


DATASET: results
Dimensiones: 49502 filas x 9 columnas
Columnas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Primeras 2 filas:
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   

    country  neutral  
0  Scotland    False  
1   England    False  

Tipos de datos:
date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

Valores nulos:
date          0
home_team     0
away_team     0
home_score    7
away_score    7
tournament    0
city          0
country       0
neutral       0
dtype: int64

DATASET: shootouts
Dimensiones: 681 filas x 5 columnas
Columnas: ['date', 'home_team', 'away

In [9]:
# ============================================================
# EXPLORACIÓN DETALLADA - UNO POR UNO
# ============================================================

for nombre, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {nombre}")
    print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Columnas: {list(df.columns)}")
    print(f"\nValores nulos por columna:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos")
    print("-"*40)




DATASET: results
Dimensiones: 49502 filas x 9 columnas
Columnas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Valores nulos por columna:
home_score    7
away_score    7
dtype: int64
----------------------------------------

DATASET: shootouts
Dimensiones: 681 filas x 5 columnas
Columnas: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']

Valores nulos por columna:
first_shooter    422
dtype: int64
----------------------------------------

DATASET: scorers
Dimensiones: 47867 filas x 8 columnas
Columnas: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']

Valores nulos por columna:
scorer     44
minute    254
dtype: int64
----------------------------------------

DATASET: names
Dimensiones: 36 filas x 4 columnas
Columnas: ['current', 'former', 'start_date', 'end_date']

Valores nulos por columna:
Sin valores nulos
----------------------------------------

DATASET: ranking1
Dimen

In [8]:
    # ============================================================
# ANÁLISIS ESPECÍFICO: PLAYERS DATASET
# Necesitamos ver exactamente qué contiene antes de decidir
# si tiene data leakage o no
# ============================================================

print("COLUMNAS PLAYERS:")
print(players.columns.tolist())

print("\nPRIMERAS 3 FILAS:")
print(players.head(3).to_string())

print("\nÚLTIMAS 3 FILAS:")
print(players.tail(3).to_string())

print("\nESTADÍSTICAS BÁSICAS:")
print(players.describe())

COLUMNAS PLAYERS:
['player_id', 'player_name', 'age', 'nationality', 'team', 'jersey_number', 'position', 'height_cm', 'weight_kg', 'preferred_foot', 'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium', 'city', 'opponent_team', 'tournament_stage', 'match_result', 'goals_team', 'goals_opponent', 'minutes_played', 'goals', 'assists', 'shots', 'shots_on_target', 'expected_goals_xg', 'expected_assists_xa', 'key_passes', 'successful_passes', 'total_passes', 'pass_accuracy', 'dribbles_attempted', 'successful_dribbles', 'crosses', 'successful_crosses', 'tackles', 'interceptions', 'clearances', 'blocks', 'aerial_duels_won', 'aerial_duels_lost', 'recoveries', 'defensive_actions', 'fouls_committed', 'fouls_suffered', 'yellow_cards', 'red_cards', 'offsides', 'saves', 'save_percentage', 'punches', 'clean_sheet', 'goals_conceded', 'penalty_saves', 'distance_covered_km', 'sprint_distance_km', 'top_speed_kmh', 'accelerations', 'decelerations', 'stamina_score', 'player_rating', 'perfo

In [11]:
# Ver datasets que se truncaron
for nombre in ["scorers", "names", "ranking1", "ranking2", "ranking3", "elo"]:
    df = datasets[nombre]
    print(f"\n{'='*60}")
    print(f"DATASET: {nombre}")
    print(f"Dimensiones: {df.shape}")
    print(f"Columnas: {list(df.columns)}")
    nulos = df.isnull().sum()
    print(f"Nulos: {nulos[nulos > 0].to_dict()}")
    print(f"\nPrimera fila:")
    print(df.head(1).to_string())


DATASET: scorers
Dimensiones: (47867, 8)
Columnas: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']
Nulos: {'scorer': 44, 'minute': 254}

Primera fila:
         date home_team away_team     team           scorer  minute  own_goal  penalty
0  1916-07-02     Chile   Uruguay  Uruguay  José Piendibene    44.0     False    False

DATASET: names
Dimensiones: (36, 4)
Columnas: ['current', 'former', 'start_date', 'end_date']
Nulos: {}

Primera fila:
  current   former  start_date    end_date
0   Benin  Dahomey  1959-11-08  1975-11-30

DATASET: ranking1
Dimensiones: (64757, 8)
Columnas: ['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']
Nulos: {}

Primera fila:
   rank country_full country_abrv  total_points  previous_points  rank_change confederation   rank_date
0     1      Germany          GER          57.0              0.0            0          UEFA  1992-12-31

DATASET: ranking2
Dime

In [19]:
# ============================================================
# ANÁLISIS DEL NUEVO DATASET DE JUGADORES
# Verificamos si es confiable antes de usarlo
# ============================================================

players_new = pd.read_csv("../data/raw/players_dataset/players.csv")

print(f"Dimensiones: {players_new.shape}")
print(f"\nColumnas: {list(players_new.columns)}")
print(f"\nPrimeras 3 filas:")
print(players_new.head(3).to_string())
print(f"\nValores nulos:")
print(players_new.isnull().sum())
print(f"\nEquipos únicos ({players_new.nunique().get('nationality', players_new.nunique().get('team', '?'))}):")

# Buscamos la columna de equipo/país
for col in players_new.columns:
    if any(x in col.lower() for x in ['team', 'nation', 'country', 'selec']):
        print(f"Columna '{col}':")
        print(players_new[col].unique())

Dimensiones: (1248, 72)

Columnas: ['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target', 'minutes_per_game', 'minutes_pct', 'minutes_per_start', 'games_complete', 'games_subs', 'minutes_per_sub', 'unused_subs', 'points_per_game', 'on_goals_for', 'on_goals_against', 'plus_minus', 'plus_minus_per90', 'plus_minus_wowy', 'cards_yellow_red', 'fouls', 'fouled', 'offsides', 'crosses', 'interceptions', 'tackles_won', 'pens_won', 'pens_conceded', 'own_goals', 'gk_games', 'gk_games_starts', 'gk_minutes', 'gk_goals_against', 'gk_goals_against_per90', 'gk_shots_on_target_against', 'gk_sa

In [20]:
# ============================================================
# ANÁLISIS DETALLADO DEL DATASET DE JUGADORES REALES
# Verificamos calidad y problemas antes de usarlo
# ============================================================

# Problema 1: formato de edad
print("Formato de edad (primeros 5):")
print(players_new['age'].head())

# Problema 2: nulos críticos
print("\nColumnas con más del 50% de nulos:")
pct_nulos = (players_new.isnull().sum() / len(players_new) * 100).round(1)
print(pct_nulos[pct_nulos > 50])

# Problema 3: equipos disponibles
print(f"\nEquipos únicos: {players_new['team_country'].nunique()}")
print(sorted(players_new['team_country'].unique()))

# Columnas más útiles para nuestro modelo
cols_utiles = ['player', 'team_country', 'position', 'age', 
               'goals', 'assists', 'minutes', 'shots_on_target_pct',
               'plus_minus', 'fouls', 'interceptions', 'tackles_won']

print("\nColumnas útiles sin nulos críticos:")
print(players_new[cols_utiles].isnull().sum())

Formato de edad (primeros 5):
0    27-022
1    23-066
2    26-141
3    24-146
4    34-258
Name: age, dtype: object

Columnas con más del 50% de nulos:
goals_per_shot_on_target       69.4
minutes_per_sub                56.1
pens_won                      100.0
pens_conceded                 100.0
gk_games                       95.2
gk_games_starts                95.2
gk_minutes                     95.2
gk_goals_against               95.2
gk_goals_against_per90         95.2
gk_shots_on_target_against     95.2
gk_saves                       95.2
gk_save_pct                    95.4
gk_wins                        95.2
gk_ties                        95.2
gk_losses                      95.2
gk_clean_sheets                95.2
gk_clean_sheets_pct            95.4
gk_pens_att                    95.2
gk_pens_allowed                95.2
gk_pens_saved                  95.2
gk_pens_missed                 95.2
gk_pens_save_pct               98.9
dtype: float64

Equipos únicos: 48
['Algeria', 'Argentina

In [21]:
# ============================================================
# LIMPIEZA DEL DATASET DE JUGADORES
# Resolvemos los problemas encontrados en el análisis
# ============================================================

import re

# Problema 1: Convertir edad "27-022" → 27
# El formato es años-dias, solo necesitamos los años
players_new['age'] = players_new['age'].apply(
    lambda x: int(str(x).split('-')[0]) if pd.notna(x) else None
)
print("Edad corregida (primeros 5):")
print(players_new['age'].head())

# Problema 2: Eliminar columnas con más del 90% de nulos
# y columnas 100% nulas que no aportan información
umbral_nulos = 0.90
cols_eliminar = [col for col in players_new.columns 
                 if players_new[col].isnull().sum() / len(players_new) > umbral_nulos]

print(f"\nColumnas eliminadas por >90% nulos ({len(cols_eliminar)}):")
print(cols_eliminar)

players_clean = players_new.drop(columns=cols_eliminar)

# Problema 3: Eliminar pens_won y pens_conceded (100% nulos)
cols_100_nulos = ['pens_won', 'pens_conceded']
players_clean = players_clean.drop(
    columns=[c for c in cols_100_nulos if c in players_clean.columns]
)

print(f"\nDimensiones después de limpieza: {players_clean.shape}")
print(f"Columnas restantes: {list(players_clean.columns)}")

Edad corregida (primeros 5):
0    27
1    23
2    26
3    24
4    34
Name: age, dtype: int64

Columnas eliminadas por >90% nulos (20):
['pens_won', 'pens_conceded', 'gk_games', 'gk_games_starts', 'gk_minutes', 'gk_goals_against', 'gk_goals_against_per90', 'gk_shots_on_target_against', 'gk_saves', 'gk_save_pct', 'gk_wins', 'gk_ties', 'gk_losses', 'gk_clean_sheets', 'gk_clean_sheets_pct', 'gk_pens_att', 'gk_pens_allowed', 'gk_pens_saved', 'gk_pens_missed', 'gk_pens_save_pct']

Dimensiones después de limpieza: (1248, 52)
Columnas restantes: ['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_s

In [22]:
# ============================================================
# AGREGACIÓN POR SELECCIÓN
# Creamos un perfil de cada equipo basado en sus jugadores
# Usamos métricas que representan la calidad ofensiva,
# defensiva y general del equipo
# ============================================================

perfil_equipos = players_clean.groupby('team_country').agg(
    # Métricas ofensivas
    goles_promedio        = ('goals', 'mean'),
    asistencias_promedio  = ('assists', 'mean'),
    shots_promedio        = ('shots', 'mean'),
    shots_target_pct      = ('shots_on_target_pct', 'mean'),
    goles_por90           = ('goals_per90', 'mean'),
    
    # Métricas defensivas
    intercepciones_prom   = ('interceptions', 'mean'),
    tackles_prom          = ('tackles_won', 'mean'),
    faltas_cometidas      = ('fouls', 'mean'),
    
    # Métricas generales
    edad_promedio         = ('age', 'mean'),
    minutos_promedio      = ('minutes', 'mean'),
    plus_minus_prom       = ('plus_minus', 'mean'),
    tarjetas_amarillas    = ('cards_yellow', 'mean'),
    
    # Cantidad de jugadores
    num_jugadores         = ('player', 'count')
).reset_index()

print(f"Perfil creado para {len(perfil_equipos)} selecciones")
print("\nPrimeras 5 selecciones:")
print(perfil_equipos.head().to_string())

print("\nEstadísticas del perfil:")
print(perfil_equipos.describe().round(2))

Perfil creado para 48 selecciones

Primeras 5 selecciones:
  team_country  goles_promedio  asistencias_promedio  shots_promedio  shots_target_pct  goles_por90  intercepciones_prom  tackles_prom  faltas_cometidas  edad_promedio  minutos_promedio  plus_minus_prom  tarjetas_amarillas  num_jugadores
0      Algeria        0.227273              0.136364        2.000000         28.658333     0.150909             1.363636      2.000000          1.318182      26.538462        180.000000        -2.045455            0.136364             26
1    Argentina        0.416667              0.208333        2.333333         33.593750     0.174167             1.708333      1.916667          1.916667      28.769231        178.750000         3.666667            0.125000             26
2    Australia        0.086957              0.043478        1.826087         31.037500     0.027391             1.260870      1.478261          2.000000      26.961538        186.521739         0.000000            0.217391     

In [23]:
# ============================================================
# GUARDAMOS PERFIL DE EQUIPOS EN PROCESSED
# Este será uno de los features de nuestro modelo
# ============================================================

perfil_equipos.to_csv(
    "../data/processed/perfil_equipos_jugadores.csv", 
    index=False
)
print("✓ Perfil de equipos guardado en data/processed/")

# ============================================================
# ANÁLISIS DEL DATASET PRINCIPAL: RESULTS
# Este es nuestro dataset más importante con 49,502 partidos
# desde 1872 hasta 2024
# ============================================================

results['date'] = pd.to_datetime(results['date'])

print("\nRango temporal:")
print(f"  Desde: {results['date'].min().date()}")
print(f"  Hasta: {results['date'].max().date()}")

print("\nTipos de torneo más frecuentes:")
print(results['tournament'].value_counts().head(10))

print("\nDistribución de resultados:")
results['resultado'] = results.apply(
    lambda x: 'Local'     if x['home_score'] > x['away_score']
    else      ('Visitante' if x['home_score'] < x['away_score']
    else       'Empate'), axis=1
)
print(results['resultado'].value_counts())
print("\nPorcentajes:")
print((results['resultado'].value_counts(normalize=True) * 100).round(1))

✓ Perfil de equipos guardado en data/processed/

Rango temporal:
  Desde: 1872-11-30
  Hasta: 2026-07-09

Tipos de torneo más frecuentes:
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1061
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

Distribución de resultados:
resultado
Local        24256
Visitante    13983
Empate       11263
Name: count, dtype: int64

Porcentajes:
resultado
Local        49.0
Visitante    28.2
Empate       22.8
Name: proportion, dtype: float64


In [24]:
# ============================================================
# DECISIONES BASADAS EN EL ANÁLISIS:
# 1. Filtramos datos hasta antes del Mundial 2026 (junio 2026)
#    para evitar data leakage
# 2. La ventaja local (49%) será un feature importante
# 3. Usaremos todos los tipos de torneo pero con peso diferente
#    (partidos oficiales pesan más que amistosos)
# ============================================================

# Filtramos partidos antes del Mundial 2026
FECHA_CORTE = '2026-06-10'  # Un día antes del inicio del Mundial 2026
results_clean = results[results['date'] < FECHA_CORTE].copy()

print(f"Partidos antes del Mundial 2026: {len(results_clean)}")
print(f"Partidos eliminados (Mundial 2026): {len(results) - len(results_clean)}")
print(f"Fecha último partido: {results_clean['date'].max().date()}")

# Verificamos distribución después del filtro
print("\nDistribución de resultados (sin Mundial 2026):")
print((results_clean['resultado'].value_counts(normalize=True) * 100).round(1))

# Agregamos peso por importancia del torneo
# Los partidos oficiales importantes deben pesar más
pesos_torneo = {
    'FIFA World Cup'                    : 4,
    'Copa América'                      : 3,
    'UEFA Euro'                         : 3,
    'African Cup of Nations'            : 3,
    'AFC Asian Cup'                     : 3,
    'FIFA World Cup qualification'      : 2,
    'UEFA Euro qualification'           : 2,
    'African Cup of Nations qualification': 2,
    'UEFA Nations League'               : 2,
    'Friendly'                          : 1,
}

results_clean['peso_torneo'] = results_clean['tournament'].map(
    lambda x: next(
        (v for k, v in pesos_torneo.items() if k in x), 1
    )
)

print("\nDistribución de pesos por torneo:")
print(results_clean.groupby('peso_torneo')['tournament'].count())

Partidos antes del Mundial 2026: 49401
Partidos eliminados (Mundial 2026): 101
Fecha último partido: 2026-06-09

Distribución de resultados (sin Mundial 2026):
resultado
Local        49.0
Visitante    28.3
Empate       22.7
Name: proportion, dtype: float64

Distribución de pesos por torneo:
peso_torneo
1    30497
2      658
3     8511
4     9735
Name: tournament, dtype: int64


In [25]:
# ============================================================
# CORRECCIÓN DE PESOS POR TORNEO
# El mapeo anterior no estaba capturando todas las variantes
# de nombres de torneos. Usamos una función más robusta.
# ============================================================

def asignar_peso(torneo):
    torneo = torneo.lower()
    if 'world cup' in torneo and 'qualif' not in torneo:
        return 4
    elif any(x in torneo for x in ['copa américa', 'euro', 
             'african cup', 'asian cup', 'gold cup', 
             'nations league', 'copa america']):
        return 3
    elif 'qualif' in torneo or 'qualification' in torneo:
        return 2
    elif 'friendly' in torneo:
        return 1
    else:
        return 2  # otros torneos oficiales

results_clean['peso_torneo'] = results_clean['tournament'].apply(asignar_peso)

print("Distribución de pesos corregida:")
dist = results_clean.groupby('peso_torneo').agg(
    cantidad    = ('tournament', 'count'),
    ejemplos    = ('tournament', lambda x: list(x.unique()[:3]))
).reset_index()
print(dist.to_string())

print(f"\nTotal partidos: {len(results_clean)}")
print(f"\nVerificación - partidos peso 4 (Mundiales):")
print(results_clean[results_clean['peso_torneo']==4]['tournament'].value_counts())

Distribución de pesos corregida:
   peso_torneo  cantidad                                                                         ejemplos
0            1     18385                                                                       [Friendly]
1            2     19596                  [British Home Championship, Évence Coppée Trophy, Muratti Vase]
2            3     10396  [Copa América, Central European International Cup, AFC Asian Cup qualification]
3            4      1024                                                 [FIFA World Cup, Viva World Cup]

Total partidos: 49401

Verificación - partidos peso 4 (Mundiales):
tournament
FIFA World Cup    964
Viva World Cup     60
Name: count, dtype: int64


In [26]:
# ============================================================
# REFINAMIENTO FINAL DE PESOS
# Basado en los datos reales encontrados
# ============================================================

def asignar_peso_v2(torneo):
    torneo_lower = torneo.lower()
    
    # Peso 4: Mundiales FIFA oficiales únicamente
    if torneo == 'FIFA World Cup':
        return 4
    
    # Peso 3: Torneos continentales principales
    elif any(x in torneo_lower for x in [
        'copa américa', 'copa america',
        'uefa euro', 'african cup of nations',
        'afc asian cup', 'gold cup',
        'uefa nations league'
    ]):
        return 3
    
    # Peso 2: Clasificatorias oficiales
    elif any(x in torneo_lower for x in [
        'qualification', 'qualifier', 'qualifying'
    ]):
        return 2
    
    # Peso 1: Amistosos y resto
    else:
        return 1

results_clean['peso_torneo'] = results_clean['tournament'].apply(asignar_peso_v2)

print("Distribución final de pesos:")
for peso in [4, 3, 2, 1]:
    subset = results_clean[results_clean['peso_torneo'] == peso]
    torneos = list(subset['tournament'].value_counts().head(3).index)
    print(f"\nPeso {peso} ({len(subset)} partidos):")
    print(f"  Ejemplos: {torneos}")

print(f"\nTotal: {len(results_clean)} partidos")

Distribución final de pesos:

Peso 4 (964 partidos):
  Ejemplos: ['FIFA World Cup']

Peso 3 (9677 partidos):
  Ejemplos: ['UEFA Euro qualification', 'African Cup of Nations qualification', 'Copa América']

Peso 2 (9852 partidos):
  Ejemplos: ['FIFA World Cup qualification', 'CFU Caribbean Cup qualification', 'CONCACAF Championship qualification']

Peso 1 (28908 partidos):
  Ejemplos: ['Friendly', 'CECAFA Cup', 'Merdeka Tournament']

Total: 49401 partidos


In [27]:
# ============================================================
# CORRECCIÓN FINAL: separar torneos principales de clasificatorias
# ============================================================

def asignar_peso_v3(torneo):
    torneo_lower = torneo.lower()
    
    # Peso 4: Mundial FIFA únicamente
    if torneo == 'FIFA World Cup':
        return 4
    
    # Peso 2: PRIMERO verificamos si es clasificatoria
    # (antes de verificar si es torneo continental)
    elif 'qualification' in torneo_lower or \
         'qualifier'     in torneo_lower or \
         'qualifying'    in torneo_lower:
        return 2
    
    # Peso 3: Torneos continentales principales (no clasificatorias)
    elif any(x in torneo_lower for x in [
        'copa américa', 'copa america',
        'uefa euro',
        'african cup of nations',
        'afc asian cup',
        'gold cup',
        'uefa nations league',
        'concacaf nations league'
    ]):
        return 3
    
    # Peso 1: Todo lo demás
    else:
        return 1

results_clean['peso_torneo'] = results_clean['tournament'].apply(asignar_peso_v3)

print("Distribución corregida:")
for peso in [4, 3, 2, 1]:
    subset = results_clean[results_clean['peso_torneo'] == peso]
    torneos = list(subset['tournament'].value_counts().head(3).index)
    print(f"\nPeso {peso} ({len(subset)} partidos):")
    print(f"  Ejemplos: {torneos}")

print(f"\nTotal: {len(results_clean)} partidos")

Distribución corregida:

Peso 4 (964 partidos):
  Ejemplos: ['FIFA World Cup']

Peso 3 (4023 partidos):
  Ejemplos: ['Copa América', 'African Cup of Nations', 'UEFA Nations League']

Peso 2 (15928 partidos):
  Ejemplos: ['FIFA World Cup qualification', 'UEFA Euro qualification', 'African Cup of Nations qualification']

Peso 1 (28486 partidos):
  Ejemplos: ['Friendly', 'CECAFA Cup', 'Merdeka Tournament']

Total: 49401 partidos


In [28]:
# ============================================================
# GUARDAMOS RESULTS LIMPIO Y CONTINUAMOS
# ============================================================

results_clean.to_csv("../data/processed/results_clean.csv", index=False)
print("✓ results_clean guardado en data/processed/")

# ============================================================
# SIGUIENTE PASO: CONSTRUIR FEATURES POR PARTIDO
# Para cada partido necesitamos calcular:
# 1. Últimos 10 partidos de cada equipo (forma reciente)
# 2. Head-to-head histórico entre los dos equipos
# 3. Elo de cada equipo antes del partido
# 4. Ranking FIFA más cercano a la fecha del partido
# Filtramos desde 1990 en adelante para datos más relevantes
# ======================================

✓ results_clean guardado en data/processed/
